<a href="https://colab.research.google.com/github/cahecaz/TrabajoFinal_analisisdesentimientos/blob/main/Analisis_Inicial_IA_Tweets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Trabajo final: analizador de sentimientos

## Etapa 1. Carga y conocimiento del dataset

**Dataset:** IA Tweets Analysis Dataset (Spanish)  
**Fuente:** Zenodo, registro 10821485  
**Objetivo de esta etapa:** cargar una copia original y comprobar su estructura, tipos de datos, valores ausentes, duplicados y distribución de las etiquetas.

> En esta etapa no se modifican los datos originales.

In [2]:
# 1. Importar la biblioteca necesaria
import pandas as pd

In [3]:
# 2. Cargar el archivo CSV
# En Google Colab, primero se debe cargar ia_tweets.csv en Archivos.
ruta_archivo = 'https://raw.githubusercontent.com/cahecaz/TrabajoFinal_analisisdesentimientos/main/ia_tweets.csv'

datos = pd.read_csv(ruta_archivo)

In [4]:
# 3. Visualizar los primeros registros
datos.head()

,ID,text,polarity,favorite_count,retweet_count,user_followers_count,user_friends_count,user_favourites_count,user_statuses_count,user_verified,user_has_extended_profile,user_is_translator,user_protected,user_default_profile
0,0,Comentaba en una charla sobre IA que el proble...,N,21,7,15936,2395,66945,68578,0,1,0,0,0
1,1,"@alvaropons Eso es imposible. Y sí, va a gener...",N,1,0,685,851,169490,128555,0,1,0,0,1
2,2,@alvaropons Lo disruptivo es que tras años de ...,NEU,1,0,685,851,169490,128555,0,1,0,0,1
3,3,"@alvaropons ¿""Prohibición del uso comercial de...",N,0,0,4877,538,6123,3156,0,1,0,0,0
4,4,@santoroydonoso Cómo se está estudiando prohib...,NEU,0,0,15936,2395,66945,68578,0,1,0,0,0


In [5]:
# 4. Conocer la cantidad de filas y columnas
print('Filas:', datos.shape[0])
print('Columnas:', datos.shape[1])

Filas: 4038
Columnas: 14


In [6]:
# 5. Conocer las columnas y sus tipos de datos
datos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4038 entries, 0 to 4037
Data columns (total 14 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   ID                         4038 non-null   int64 
 1   text                       4038 non-null   object
 2   polarity                   4038 non-null   object
 3   favorite_count             4038 non-null   int64 
 4   retweet_count              4038 non-null   int64 
 5   user_followers_count       4038 non-null   int64 
 6   user_friends_count         4038 non-null   int64 
 7   user_favourites_count      4038 non-null   int64 
 8   user_statuses_count        4038 non-null   int64 
 9   user_verified              4038 non-null   int64 
 10  user_has_extended_profile  4038 non-null   int64 
 11  user_is_translator         4038 non-null   int64 
 12  user_protected             4038 non-null   int64 
 13  user_default_profile       4038 non-null   int64 
dtypes: int64

In [7]:
# 6. Contabilizar valores ausentes
datos.isnull().sum()

,0
ID,0
text,0
polarity,0
favorite_count,0
retweet_count,0
user_followers_count,0
user_friends_count,0
user_favourites_count,0
user_statuses_count,0
user_verified,0


In [8]:
# 7. Contabilizar filas duplicadas y textos repetidos
print('Filas completamente duplicadas:', datos.duplicated().sum())
print('Textos repetidos:', datos['text'].duplicated().sum())

Filas completamente duplicadas: 0
Textos repetidos: 57


In [9]:
# 8. Examinar la distribución de las etiquetas
distribucion = datos['polarity'].value_counts()
distribucion

,count
polarity,
NEU,2689
P,757
N,592


In [10]:
# 9. Mostrar la distribución porcentual
porcentaje = datos['polarity'].value_counts(normalize=True).mul(100).round(2)
porcentaje

,proportion
polarity,
NEU,66.59
P,18.75
N,14.66


## Resultados comprobados

- El dataset contiene **4.038 registros y 14 columnas**.
- No presenta valores ausentes ni textos vacíos.
- Los 4.038 identificadores son únicos.
- No existen filas completamente duplicadas, pero hay **57 textos repetidos**.
- La distribución es: **2.689 neutros (66,59 %), 757 positivos (18,75 %) y 592 negativos (14,66 %)**.
- La clase neutra predomina, por lo que el desequilibrio deberá considerarse al dividir los datos y al interpretar precisión, recall y F1-score.
- La decisión sobre el tratamiento de los 57 textos repetidos se adoptará en la etapa de preparación, manteniendo intacto el archivo original.

## Sustento y referencias de esta etapa

La fuente original documenta la procedencia, estructura y uso previsto del dataset (Guerrero-Contreras et al., 2024a). El estudio asociado analiza sentimientos en redes sociales mediante texto y metadatos (Guerrero-Contreras et al., 2024b). Para el posterior análisis del rendimiento por clase se considera que una sola medida puede ocultar diferencias entre categorías, por lo que se conservarán precisión, recall y F1-score por clase (Sokolova & Lapalme, 2009).

### Referencias APA 7

Guerrero-Contreras, G., Balderas-Díaz, S., Serrano-Fernández, A., & Muñoz, A. (2024a). *IA Tweets Analysis Dataset (Spanish)* (Version v1) [Data set]. Zenodo. https://doi.org/10.5281/zenodo.10821485

Guerrero-Contreras, G., Balderas-Díaz, S., Serrano-Fernández, A., & Muñoz, A. (2024b). Enhancing sentiment analysis on social media: Integrating text and metadata for refined insights. En *2024 International Conference on Intelligent Environments (IE)* (pp. 62–69). IEEE. https://doi.org/10.1109/IE61493.2024.10599899

Sokolova, M., & Lapalme, G. (2009). A systematic analysis of performance measures for classification tasks. *Information Processing & Management, 45*(4), 427–437. https://doi.org/10.1016/j.ipm.2009.03.002

# Etapa 2. Preparación y limpieza de los textos

## Diagnóstico previo

Antes de modificar los textos, se examina la presencia de enlaces, menciones,
hashtags, saltos de línea y espacios repetidos. El texto original se conservará
en la columna `text`, mientras que las transformaciones posteriores se almacenarán en una nueva columna llamada `texto_limpio`.

In [11]:
# 10. Identificar elementos presentes antes de la limpieza

cantidad_registros = len(datos)

diagnostico_textos = pd.DataFrame({
    'Elemento': [
        'Enlaces',
        'Menciones',
        'Hashtags',
        'Saltos de línea',
        'Espacios repetidos'
    ],
    'Cantidad': [
        datos['text'].str.contains(
            r'https?://\S+|www\.\S+',
            regex=True,
            na=False
        ).sum(),

        datos['text'].str.contains(
            r'@\w+',
            regex=True,
            na=False
        ).sum(),

        datos['text'].str.contains(
            r'#\w+',
            regex=True,
            na=False
        ).sum(),

        datos['text'].str.contains(
            '\n',
            regex=False,
            na=False
        ).sum(),

        datos['text'].str.contains(
            r'\s{2,}',
            regex=True,
            na=False
        ).sum()
    ]
})

diagnostico_textos['Porcentaje'] = (
    diagnostico_textos['Cantidad']
    / cantidad_registros
    * 100
).round(2)

diagnostico_textos

,Elemento,Cantidad,Porcentaje
0,Enlaces,1308,32.39
1,Menciones,2104,52.11
2,Hashtags,616,15.26
3,Saltos de línea,1182,29.27
4,Espacios repetidos,1084,26.84


## Sustento académico de la etapa

El preprocesamiento puede modificar el rendimiento de un clasificador, por lo
que las transformaciones deben definirse según las características reales del
dataset. En textos procedentes de redes sociales, elementos como hashtags,
emojis y expresiones informales pueden contener información relevante para el
sentimiento. Por esta razón, primero se diagnostica su presencia y posteriormente
se determina su tratamiento.

### Referencias APA 7

Guerrero-Contreras, G., Balderas-Díaz, S., Serrano-Fernández, A., y Muñoz, A.
(2024). Enhancing sentiment analysis on social media: Integrating text and
metadata for refined insights. En *2024 International Conference on Intelligent
Environments (IE)* (pp. 62–69). IEEE.
https://doi.org/10.1109/IE61493.2024.10599899

Krouska, A., Troussas, C., y Virvou, M. (2016). The effect of preprocessing
techniques on Twitter sentiment analysis. En *2016 7th International Conference
on Information, Intelligence, Systems & Applications (IISA)* (pp. 1–5). IEEE.
https://doi.org/10.1109/IISA.2016.7785373

Symeonidis, S., Effrosynidis, D., y Arampatzis, A. (2018). A comparative
evaluation of pre-processing techniques and their interactions for Twitter
sentiment analysis. *Expert Systems with Applications, 110*, 298–310.
https://doi.org/10.1016/j.eswa.2018.06.022

In [12]:
# 11. Crear y limpiar la columna de texto

# Conservar intacta la columna original
datos['texto_limpio'] = datos['text'].copy()

# Convertir el texto a minúsculas
datos['texto_limpio'] = datos['texto_limpio'].str.lower()

# Eliminar enlaces
datos['texto_limpio'] = datos['texto_limpio'].str.replace(
    r'https?://\S+|www\.\S+',
    '',
    regex=True
)

# Eliminar menciones
datos['texto_limpio'] = datos['texto_limpio'].str.replace(
    r'@\w+',
    '',
    regex=True
)

# Eliminar el signo #, conservando la palabra del hashtag
datos['texto_limpio'] = datos['texto_limpio'].str.replace(
    '#',
    '',
    regex=False
)

# Reemplazar saltos de línea y espacios repetidos
datos['texto_limpio'] = datos['texto_limpio'].str.replace(
    r'\s+',
    ' ',
    regex=True
)

# Eliminar espacios al comienzo y al final
datos['texto_limpio'] = datos['texto_limpio'].str.strip()

In [13]:
# 12. Comprobar el resultado de la limpieza

print(
    'Textos vacíos después de la limpieza:',
    datos['texto_limpio'].eq('').sum()
)

print(
    'Enlaces restantes:',
    datos['texto_limpio'].str.contains(
        r'https?://\S+|www\.\S+',
        regex=True,
        na=False
    ).sum()
)

print(
    'Menciones restantes:',
    datos['texto_limpio'].str.contains(
        r'@\w+',
        regex=True,
        na=False
    ).sum()
)

print(
    'Signos # restantes:',
    datos['texto_limpio'].str.contains(
        '#',
        regex=False,
        na=False
    ).sum()
)

print(
    'Saltos de línea restantes:',
    datos['texto_limpio'].str.contains(
        '\n',
        regex=False,
        na=False
    ).sum()
)

print(
    'Espacios repetidos restantes:',
    datos['texto_limpio'].str.contains(
        r'\s{2,}',
        regex=True,
        na=False
    ).sum()
)

print(
    'Textos repetidos después de la limpieza:',
    datos['texto_limpio'].duplicated().sum()
)

datos[['text', 'texto_limpio']].head(10)

Textos vacíos después de la limpieza: 52
Enlaces restantes: 0
Menciones restantes: 0
Signos # restantes: 0
Saltos de línea restantes: 0
Espacios repetidos restantes: 0
Textos repetidos después de la limpieza: 140


,text,texto_limpio
0,Comentaba en una charla sobre IA que el proble...,comentaba en una charla sobre ia que el proble...
1,"@alvaropons Eso es imposible. Y sí, va a gener...","eso es imposible. y sí, va a generar un montón..."
2,@alvaropons Lo disruptivo es que tras años de ...,lo disruptivo es que tras años de decir que la...
3,"@alvaropons ¿""Prohibición del uso comercial de...","¿""prohibición del uso comercial de las imágene..."
4,@santoroydonoso Cómo se está estudiando prohib...,cómo se está estudiando prohibir chatgpt en eu...
5,@santoroydonoso @alvaropons Mientras tanto en ...,mientras tanto en sadaic
6,Esta transición va a estar bien difícil... El ...,esta transición va a estar bien difícil... el ...
7,Va a costar mucho adaptarse a los nuevos cambi...,va a costar mucho adaptarse a los nuevos cambi...
8,8/ Conclusión: La inteligencia artificial ha l...,8/ conclusión: la inteligencia artificial ha l...
9,9/ ¿Te gustó este hilo? ¡Comparte y sígueme pa...,9/ ¿te gustó este hilo? ¡comparte y sígueme pa...


In [14]:
# 13. Examinar textos vacíos y duplicados después de la limpieza

textos_vacios = datos[
    datos['texto_limpio'].eq('')
][['text', 'polarity', 'texto_limpio']]

textos_duplicados = datos[
    datos['texto_limpio'].duplicated(keep=False)
][['text', 'polarity', 'texto_limpio']].sort_values(
    by='texto_limpio'
)

# Identificar textos iguales con diferentes etiquetas
cantidad_etiquetas = textos_duplicados.groupby(
    'texto_limpio'
)['polarity'].nunique()

textos_con_conflicto = cantidad_etiquetas[
    cantidad_etiquetas > 1
]

print('Registros con texto vacío:', len(textos_vacios))

print(
    'Registros involucrados en duplicados:',
    len(textos_duplicados)
)

print(
    'Textos duplicados con etiquetas diferentes:',
    len(textos_con_conflicto)
)

print('\nEjemplos de textos originales que quedaron vacíos:')
display(textos_vacios.head(10))

print('\nEjemplos de textos repetidos después de la limpieza:')
display(textos_duplicados.head(20))

Registros con texto vacío: 52
Registros involucrados en duplicados: 174
Textos duplicados con etiquetas diferentes: 1

Ejemplos de textos originales que quedaron vacíos:


,text,polarity,texto_limpio
110,@Chuskitin https://t.co/PBfWvTNlXr,NEU,
131,@lem_antonieta,NEU,
308,@EdelbertoJose https://t.co/OW2lqPz1UC,NEU,
318,@LeonKrauze @nytimes https://t.co/xVfL4eTxo0,NEU,
389,@Mariaporia @MJDuzan https://t.co/WAvpP6Vt8e,NEU,
476,@isabel_iglesias https://t.co/JfR45iKlCB,NEU,
568,@LaBrujulaXR,NEU,
645,@_d3lm0nt https://t.co/ewQQ84hVW3,NEU,
740,@RTVCComunica @SergioMiro https://t.co/u6krqhO...,NEU,
818,@AnimeLtd @f3_kenshin https://t.co/I9l736ll8s,NEU,



Ejemplos de textos repetidos después de la limpieza:


,text,polarity,texto_limpio
1362,@elpais_tec https://t.co/174AyIvavP,NEU,
1178,@carlotagalvan @Unancorcom https://t.co/TEcV1x...,NEU,
1218,@RosanaFerrero https://t.co/brHsjo2ufm,NEU,
1219,@RosanaFerrero https://t.co/eWw4Zvw0AB,NEU,
1329,@alba_delcampo https://t.co/eSwHpofjNK,NEU,
3731,@ViajesDigitales @lasseweb20 https://t.co/MDKF...,NEU,
1377,@marcsabaletee https://t.co/aL1mxBT8rQ,NEU,
3495,@beincrypto_es https://t.co/Sg6FiGcCAL,NEU,
2002,@ivandmattar https://t.co/47ucvQTnxI,NEU,
1446,@Tomi_dep @SoyReyMidas https://t.co/CYfYSvfTql,NEU,


In [15]:
# 14. Separar los textos vacíos y examinar el conflicto de etiquetas

# Distribución de las etiquetas de los textos vacíos
print('Etiquetas de los textos vacíos:')
print(textos_vacios['polarity'].value_counts())

# Crear una selección temporal sin textos vacíos
datos_no_vacios = datos[
    datos['texto_limpio'].ne('')
].copy()

print(
    '\nRegistros disponibles después de excluir los vacíos:',
    len(datos_no_vacios)
)

print(
    'Duplicados adicionales sin considerar los vacíos:',
    datos_no_vacios['texto_limpio'].duplicated().sum()
)

print(
    'Registros involucrados en duplicados no vacíos:',
    datos_no_vacios['texto_limpio'].duplicated(
        keep=False
    ).sum()
)

# Mostrar el texto duplicado que posee etiquetas diferentes
registros_con_conflicto = datos_no_vacios[
    datos_no_vacios['texto_limpio'].isin(
        textos_con_conflicto.index
    )
][['text', 'polarity', 'texto_limpio']]

print('\nTexto duplicado con etiquetas diferentes:')
display(registros_con_conflicto)

Etiquetas de los textos vacíos:
polarity
NEU    52
Name: count, dtype: int64

Registros disponibles después de excluir los vacíos: 3986
Duplicados adicionales sin considerar los vacíos: 89
Registros involucrados en duplicados no vacíos: 122

Texto duplicado con etiquetas diferentes:


,text,polarity,texto_limpio
1052,@aalbaperez Totalmente de acuerdo,P,totalmente de acuerdo
2072,@mjmolano @almaldo2 Totalmente de acuerdo,NEU,totalmente de acuerdo
2882,@DrNickolaz Totalmente de acuerdo,P,totalmente de acuerdo
3590,@rforcano Totalmente de acuerdo,P,totalmente de acuerdo


In [16]:
# 15. Crear el conjunto de datos limpio

# Identificar textos repetidos con etiquetas diferentes
cantidad_etiquetas = datos_no_vacios.groupby(
    'texto_limpio'
)['polarity'].nunique()

textos_conflictivos = cantidad_etiquetas[
    cantidad_etiquetas > 1
].index

# Excluir los textos con etiquetas contradictorias
datos_limpios = datos_no_vacios[
    ~datos_no_vacios['texto_limpio'].isin(
        textos_conflictivos
    )
].copy()

# Conservar una sola aparición de cada texto repetido
datos_limpios = datos_limpios.drop_duplicates(
    subset='texto_limpio',
    keep='first'
)

# Reorganizar el índice
datos_limpios = datos_limpios.reset_index(drop=True)

print('Registros originales:', len(datos))
print('Registros del conjunto limpio:', len(datos_limpios))
print('Registros excluidos:', len(datos) - len(datos_limpios))

print(
    'Textos vacíos restantes:',
    datos_limpios['texto_limpio'].eq('').sum()
)

print(
    'Textos duplicados restantes:',
    datos_limpios['texto_limpio'].duplicated().sum()
)

print('\nDistribución final de sentimientos:')
print(datos_limpios['polarity'].value_counts())

datos_limpios[['text', 'texto_limpio', 'polarity']].head(10)

Registros originales: 4038
Registros del conjunto limpio: 3896
Registros excluidos: 142
Textos vacíos restantes: 0
Textos duplicados restantes: 0

Distribución final de sentimientos:
polarity
NEU    2608
P       700
N       588
Name: count, dtype: int64


,text,texto_limpio,polarity
0,Comentaba en una charla sobre IA que el proble...,comentaba en una charla sobre ia que el proble...,N
1,"@alvaropons Eso es imposible. Y sí, va a gener...","eso es imposible. y sí, va a generar un montón...",N
2,@alvaropons Lo disruptivo es que tras años de ...,lo disruptivo es que tras años de decir que la...,NEU
3,"@alvaropons ¿""Prohibición del uso comercial de...","¿""prohibición del uso comercial de las imágene...",N
4,@santoroydonoso Cómo se está estudiando prohib...,cómo se está estudiando prohibir chatgpt en eu...,NEU
5,@santoroydonoso @alvaropons Mientras tanto en ...,mientras tanto en sadaic,NEU
6,Esta transición va a estar bien difícil... El ...,esta transición va a estar bien difícil... el ...,N
7,Va a costar mucho adaptarse a los nuevos cambi...,va a costar mucho adaptarse a los nuevos cambi...,N
8,8/ Conclusión: La inteligencia artificial ha l...,8/ conclusión: la inteligencia artificial ha l...,P
9,9/ ¿Te gustó este hilo? ¡Comparte y sígueme pa...,9/ ¿te gustó este hilo? ¡comparte y sígueme pa...,P


## Conclusión de la limpieza

El proceso de limpieza generó un conjunto de 3.896 registros utilizables. Se
excluyeron 142 registros, equivalentes al 3,52 % del dataset original.

La exclusión consideró textos que quedaron vacíos después de retirar enlaces y
menciones, duplicados generados o identificados durante la normalización y un
texto repetido que presentaba etiquetas de sentimiento contradictorias.

La columna `text` conserva el contenido original, mientras que `texto_limpio`
contiene la versión preparada para las siguientes etapas. El conjunto final no
presenta textos vacíos ni duplicados.

La categoría neutra continúa siendo predominante, con el 66,94 % de los
registros. Esta distribución deberá considerarse posteriormente al dividir los
datos y al interpretar las métricas del clasificador.